# One-Particle Block-Encoding Check with F Tensor

This notebook builds the one-particle sparse block-encoding from an `F` tensor table and verifies the encoded block against the truncated full `F_{pq}` matrix.

In [11]:
from pathlib import Path
import sys

import numpy as np
import cirq
from qualtran.cirq_interop import BloqAsCirqGate, get_named_qubits

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'experiments' else Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from exciton.benchmark_tensors import generate_f_tensor
from block_encoding.two_particle_row_oracles import build_one_particle_sparse_block_encoding

In [20]:
# Parameters
D = 1
L = 8
R_loc = 1
a = 1.0
entry_bitsize = 20
metric = 'manhattan'

# Oracle-table input M(i,m): shape (L,)^D + (2R_loc+1,)^D
M = generate_f_tensor(
    L=L, D=D, a=a, metric=metric, r_cut=R_loc, oracle_convention='row_oracle'
)

# Full reference F_{pq} with same hard cutoff
F_ref = generate_f_tensor(
    L=L, D=D, a=a, metric=metric, r_cut=R_loc, oracle_convention='pq'
)

bloq = build_one_particle_sparse_block_encoding(
    M=M, D=D, L=L, R_loc=R_loc, entry_bitsize=entry_bitsize
)

print('M shape:', M.shape)
print('F_ref shape:', F_ref.shape)
print('bloq:', type(bloq).__name__)
print('signature:', bloq.signature)

M shape: (8, 3)
F_ref shape: (8, 8)
bloq: OneParticleSparseBlockEncoding
signature: Signature((Register(name='q', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='i', dtype=BQUInt(bitsize=3, iteration_length=8), _shape=(1,), side=<Side.THRU: 3>)))


In [19]:
# Dense unitary (small demo only)
named = get_named_qubits(bloq.signature.lefts())
qorder = []
reg_qubits = {}
for reg in bloq.signature.lefts():
    arr = np.array(named[reg.name], dtype=object).reshape(-1)
    reg_qubits[reg.name] = arr
    qorder.extend(arr.tolist())

U = cirq.unitary(BloqAsCirqGate(bloq).on(*qorder))
print('U shape:', U.shape)

U shape: (64, 64)


In [14]:
def bits_of(val: int, n: int):
    return [int((val >> (n - 1 - k)) & 1) for k in range(n)]

def basis_index(q_val: int, m_val: int, i_val: int) -> int:
    bits = []
    bits.extend(bits_of(q_val, len(reg_qubits['q'])))
    bits.extend(bits_of(m_val, len(reg_qubits['m'])))
    bits.extend(bits_of(i_val, len(reg_qubits['i'])))
    return int(cirq.big_endian_bits_to_int(bits))

# Extract postselected block B = <0_q,0_m|U|0_q,0_m> on i-register
B = np.zeros((L, L), dtype=np.complex128)
for i_out in range(L):
    for i_in in range(L):
        r = basis_index(0, 0, i_out)
        c = basis_index(0, 0, i_in)
        B[i_out, i_in] = U[r, c]

# For this construction, the encoded matrix carries normalization 1 / (2R_loc+1)^D.
scale = (2 * R_loc + 1) ** D
B_scaled = scale * B

err = np.max(np.abs(B_scaled - F_ref))
print('scale =', scale)
print('max abs error (scaled block vs F_ref):', err)
print('B_scaled:', np.real_if_close(B_scaled))
print('F_ref:', F_ref)
assert np.allclose(B_scaled, F_ref, atol=1e-4)

scale = 3
max abs error (scaled block vs F_ref): 3.7390445450924403e-07
B_scaled: [[1.00000000e+00 3.67879067e-01 7.77474334e-33 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [3.67879067e-01 1.00000000e+00 3.67879067e-01 7.77474334e-33
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 3.67879067e-01 1.00000000e+00 3.67879067e-01
  7.77474334e-33 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 3.67879067e-01 1.00000000e+00
  3.67879067e-01 7.77474334e-33 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 3.67879067e-01
  1.00000000e+00 3.67879067e-01 7.77474334e-33 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  3.67879067e-01 1.00000000e+00 3.67879067e-01 7.77474334e-33]
 [7.77474334e-33 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 3.67879067e-01 1.00000000e+00 3.67879067e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e